In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

WD = Path(
    "/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/"
    "7_HIPHOP_validation/Single-dose_QTL_randomized"
)

RAW_DIR = WD / "raw_data"
INPUT_DIR = WD / "input"

SAMPLE_KEY_FILE = (
    RAW_DIR / "sample_key.tsv"
)

# True: the primary OD600 column is blank-corrected.
# False: the primary OD600 column contains raw readings.
APPLY_BLANK_CORRECTION = True

BLANK_LABELS = {
    "blank",
    "blk",
    "empty",
}

if not RAW_DIR.is_dir():
    raise FileNotFoundError(
        f"Raw-data directory does not exist: {RAW_DIR}"
    )

if not SAMPLE_KEY_FILE.is_file():
    raise FileNotFoundError(
        f"Sample key does not exist: {SAMPLE_KEY_FILE}"
    )

INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Working directory: {WD}")
print(f"Raw data:         {RAW_DIR}")
print(f"Output:           {INPUT_DIR}")

Working directory: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized
Raw data:         /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/raw_data
Output:           /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input


In [2]:
def load_sample_key(path):
    key = pd.read_csv(
        path,
        sep="\t",
        header=None,
        dtype=str,
        comment="#",
    )

    key = (
        key.dropna(how="all")
        .dropna(axis=1, how="all")
    )

    if key.shape[1] == 2:
        key.columns = [
            "sample_id",
            "gene",
        ]

        key["qtl"] = "all"

    elif key.shape[1] == 3:
        key.columns = [
            "sample_id",
            "gene",
            "qtl",
        ]

    else:
        raise ValueError(
            "sample_key.tsv must contain either:\n"
            "  2 columns: sample_id, gene\n"
            "or\n"
            "  3 columns: sample_id, gene, qtl"
        )

    required_columns = [
        "sample_id",
        "gene",
        "qtl",
    ]

    for column in required_columns:
        key[column] = (
            key[column]
            .astype("string")
            .str.strip()
        )

    missing_values = (
        key[required_columns].isna().any().any()
        or (key[required_columns] == "").any().any()
    )

    if missing_values:
        raise ValueError(
            "Every sample-key row must contain "
            "sample_id, gene and qtl."
        )

    if key["sample_id"].duplicated().any():
        duplicated = (
            key.loc[
                key["sample_id"].duplicated(False),
                "sample_id",
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            "Duplicate sample IDs in sample_key.tsv: "
            f"{duplicated}"
        )

    return key.reset_index(drop=True)


sample_key = load_sample_key(
    SAMPLE_KEY_FILE
)

sample_to_gene = (
    sample_key
    .set_index("sample_id")["gene"]
    .to_dict()
)

sample_to_qtl = (
    sample_key
    .set_index("sample_id")["qtl"]
    .to_dict()
)

display(sample_key)

print(
    f"Loaded {len(sample_key)} sample mappings "
    f"from {sample_key['qtl'].nunique()} QTL group(s)."
)

,sample_id,gene,qtl
0,1,TIF11,all
1,2,TRM732,all
2,3,YMR262W,all
3,4,BUL1,all
4,5,PPA2,all
5,6,RRN9,all
6,7,SCS7,all
7,8,ZDS1,all
8,9,URA10,all
9,10,RSN1,all


Loaded 10 sample mappings from 1 QTL group(s).


In [3]:
def infer_condition(plate_id):
    """
    Extract the final {N}x condition from a plate name.

    Examples
    --------
    CHR8_C_0x   -> 0x
    CHR8_T_1x   -> 1x
    CHR8_T_2x   -> 2x
    CHR8_T_10x  -> 10x
    CHR8_T_0.5x -> 0.5x
    """
    match = re.search(
        r"(?:^|_)(\d+(?:\.\d+)?)x$",
        plate_id,
        flags=re.IGNORECASE,
    )

    if match is None:
        raise ValueError(
            "Could not extract a terminal {N}x condition "
            f"from plate name: {plate_id}"
        )

    return f"{match.group(1)}x".lower()


def condition_number(condition):
    return float(
        condition.lower().removesuffix("x")
    )


map_files = sorted(
    RAW_DIR.glob("*_map.csv")
)

if not map_files:
    raise FileNotFoundError(
        f"No *_map.csv files found in {RAW_DIR}"
    )

plate_pairs = []
missing_raw_files = []

for map_file in map_files:
    plate_id = map_file.name[
        :-len("_map.csv")
    ]

    raw_file = map_file.with_name(
        f"{plate_id}.csv"
    )

    if raw_file.is_file():
        plate_pairs.append(
            (map_file, raw_file)
        )
    else:
        missing_raw_files.append(
            raw_file.name
        )

if missing_raw_files:
    raise FileNotFoundError(
        "Plate maps without matching raw OD files: "
        + ", ".join(missing_raw_files)
    )

pair_records = []

for map_file, raw_file in plate_pairs:
    condition = infer_condition(
        raw_file.stem
    )

    pair_records.append({
        "plate": raw_file.stem,
        "condition": condition,
        "condition_number": condition_number(
            condition
        ),
        "plate_map": map_file.name,
        "raw_OD": raw_file.name,
    })

pairs_table = (
    pd.DataFrame(pair_records)
    .sort_values(
        [
            "condition_number",
            "plate",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

display(pairs_table)

print(
    "Conditions discovered:",
    pairs_table["condition"]
    .drop_duplicates()
    .tolist(),
)

,plate,condition,condition_number,plate_map,raw_OD
0,CHR13_C_0x,0x,0.0,CHR13_C_0x_map.csv,CHR13_C_0x.csv
1,CHR13_T_1x,1x,1.0,CHR13_T_1x_map.csv,CHR13_T_1x.csv
2,CHR13_T_2x,2x,2.0,CHR13_T_2x_map.csv,CHR13_T_2x.csv


Conditions discovered: ['0x', '1x', '2x']


In [4]:
def read_plate_matrix(
    path,
    *,
    numeric=False,
):
    plate = pd.read_csv(
        path,
        index_col=0,
        dtype=None if numeric else str,
    )

    # The first CSV row and column are plate coordinates.
    plate.index = (
        plate.index
        .astype(str)
        .str.strip()
        .str.upper()
    )

    plate.columns = (
        plate.columns
        .astype(str)
        .str.strip()
    )

    if plate.index.has_duplicates:
        raise ValueError(
            f"Duplicate row coordinates in {path.name}"
        )

    if plate.columns.has_duplicates:
        raise ValueError(
            f"Duplicate column coordinates in {path.name}"
        )

    if numeric:
        plate = plate.apply(
            pd.to_numeric,
            errors="coerce",
        )
    else:
        plate = plate.apply(
            lambda column: column.str.strip()
        )

    return plate


def plate_to_long(
    plate,
    value_name,
):
    # Produces row-major well order:
    # A1, A2, ..., A12, B1, ..., H12.
    long = pd.DataFrame({
        "row": np.repeat(
            plate.index.to_numpy(),
            len(plate.columns),
        ),
        "column": np.tile(
            plate.columns.to_numpy(),
            len(plate.index),
        ),
        value_name: (
            plate.to_numpy()
            .reshape(-1)
        ),
    })

    long["well"] = (
        long["row"]
        + long["column"]
    )

    return long


def process_plate(
    map_file,
    raw_file,
    sample_key_df,
):
    plate_id = raw_file.stem

    condition = infer_condition(
        plate_id
    )

    plate_map = read_plate_matrix(
        map_file,
        numeric=False,
    )

    raw_od = read_plate_matrix(
        raw_file,
        numeric=True,
    )

    if not plate_map.index.equals(
        raw_od.index
    ):
        raise ValueError(
            "Row coordinates differ between "
            f"{map_file.name} and {raw_file.name}"
        )

    if not plate_map.columns.equals(
        raw_od.columns
    ):
        raise ValueError(
            "Column coordinates differ between "
            f"{map_file.name} and {raw_file.name}"
        )

    map_long = plate_to_long(
        plate_map,
        "sample_id",
    )

    od_long = plate_to_long(
        raw_od,
        "OD600_raw",
    )

    plate = map_long.merge(
        od_long[
            [
                "row",
                "column",
                "OD600_raw",
            ]
        ],
        on=[
            "row",
            "column",
        ],
        how="left",
        validate="one_to_one",
    )

    plate["sample_id"] = (
        plate["sample_id"]
        .astype("string")
        .str.strip()
    )

    missing_labels = (
        plate["sample_id"].isna()
        | (plate["sample_id"] == "")
    )

    if missing_labels.any():
        bad_wells = plate.loc[
            missing_labels,
            "well",
        ].tolist()

        raise ValueError(
            f"Unlabelled wells in {map_file.name}: "
            f"{bad_wells}"
        )

    if plate["OD600_raw"].isna().any():
        bad_wells = plate.loc[
            plate["OD600_raw"].isna(),
            "well",
        ].tolist()

        raise ValueError(
            "Missing or non-numeric OD values in "
            f"{raw_file.name}: {bad_wells}"
        )

    plate["is_blank"] = (
        plate["sample_id"]
        .str.lower()
        .isin(BLANK_LABELS)
    )

    blank_wells = plate.loc[
        plate["is_blank"]
    ].copy()

    if blank_wells.empty:
        raise ValueError(
            f"No blank wells identified in {map_file.name}"
        )

    blank_mean = (
        blank_wells["OD600_raw"].mean()
    )

    blank_median = (
        blank_wells["OD600_raw"].median()
    )

    blank_sd = (
        blank_wells["OD600_raw"].std(ddof=1)
    )

    measurements = plate.loc[
        ~plate["is_blank"]
    ].copy()

    known_ids = set(
        sample_key_df["sample_id"]
    )

    observed_ids = set(
        measurements["sample_id"]
    )

    unknown_ids = sorted(
        observed_ids - known_ids
    )

    if unknown_ids:
        raise ValueError(
            f"Sample IDs in {map_file.name} are "
            "absent from sample_key.tsv: "
            f"{unknown_ids}"
        )

    measurements = measurements.merge(
        sample_key_df,
        on="sample_id",
        how="left",
        validate="many_to_one",
    )

    measurements["replicate_number"] = (
        measurements
        .groupby(
            "sample_id",
            sort=False,
        )
        .cumcount()
        + 1
    )

    max_replicate = int(
        measurements[
            "replicate_number"
        ].max()
    )

    replicate_width = max(
        2,
        len(str(max_replicate)),
    )

    measurements["replicate"] = (
        measurements[
            "replicate_number"
        ]
        .map(
            lambda number: (
                f"replicate_"
                f"{number:0{replicate_width}d}"
            )
        )
    )

    measurements["plate"] = plate_id
    measurements["condition"] = condition
    measurements["blank_mean"] = blank_mean

    measurements[
        "OD600_blank_corrected"
    ] = (
        measurements["OD600_raw"]
        - blank_mean
    )

    if APPLY_BLANK_CORRECTION:
        measurements["OD600"] = (
            measurements[
                "OD600_blank_corrected"
            ]
        )
    else:
        measurements["OD600"] = (
            measurements["OD600_raw"]
        )

    output_columns = [
        "plate",
        "condition",
        "sample_id",
        "gene",
        "qtl",
        "replicate",
        "replicate_number",
        "well",
        "OD600",
        "OD600_raw",
        "blank_mean",
        "OD600_blank_corrected",
    ]

    measurements = (
        measurements[output_columns]
        .sort_values(
            [
                "sample_id",
                "replicate_number",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    blank_wells = blank_wells.assign(
        plate=plate_id,
        condition=condition,
    )

    blank_wells = (
        blank_wells[
            [
                "plate",
                "condition",
                "well",
                "OD600_raw",
            ]
        ]
        .reset_index(drop=True)
    )

    blank_summary = pd.DataFrame([
        {
            "plate": plate_id,
            "condition": condition,
            "blank_n": len(blank_wells),
            "blank_mean": blank_mean,
            "blank_median": blank_median,
            "blank_sd": blank_sd,
            "blank_min": (
                blank_wells["OD600_raw"].min()
            ),
            "blank_max": (
                blank_wells["OD600_raw"].max()
            ),
        }
    ])

    return (
        measurements,
        blank_wells,
        blank_summary,
    )

In [5]:
all_long = []
all_blank_wells = []
all_blank_summaries = []
written_files = []

for map_file, raw_file in plate_pairs:
    (
        long_df,
        blank_wells_df,
        blank_summary_df,
    ) = process_plate(
        map_file,
        raw_file,
        sample_key,
    )

    plate_id = raw_file.stem

    long_path = (
        INPUT_DIR
        / f"{plate_id}_input_long.tsv"
    )

    long_df.to_csv(
        long_path,
        sep="\t",
        index=False,
        float_format="%.10g",
    )

    written_files.append(
        long_path
    )

    all_long.append(
        long_df
    )

    all_blank_wells.append(
        blank_wells_df
    )

    all_blank_summaries.append(
        blank_summary_df
    )

combined_long = pd.concat(
    all_long,
    ignore_index=True,
)

combined_blank_wells = pd.concat(
    all_blank_wells,
    ignore_index=True,
)

blank_summary = pd.concat(
    all_blank_summaries,
    ignore_index=True,
)

# Order combined results numerically by dose.
condition_order = (
    pairs_table[
        [
            "condition",
            "condition_number",
        ]
    ]
    .drop_duplicates()
    .sort_values("condition_number")
    ["condition"]
    .tolist()
)

combined_long["condition"] = pd.Categorical(
    combined_long["condition"],
    categories=condition_order,
    ordered=True,
)

combined_long = (
    combined_long
    .sort_values(
        [
            "condition",
            "plate",
            "qtl",
            "sample_id",
            "replicate_number",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Convert back to ordinary strings before TSV export.
combined_long["condition"] = (
    combined_long["condition"]
    .astype("string")
)

combined_long_path = (
    INPUT_DIR
    / "all_plates_input_long.tsv"
)

combined_long.to_csv(
    combined_long_path,
    sep="\t",
    index=False,
    float_format="%.10g",
)

written_files.append(
    combined_long_path
)

blank_wells_path = (
    INPUT_DIR
    / "all_blank_wells.tsv"
)

blank_summary_path = (
    INPUT_DIR
    / "blank_summary.tsv"
)

combined_blank_wells.to_csv(
    blank_wells_path,
    sep="\t",
    index=False,
    float_format="%.10g",
)

blank_summary.to_csv(
    blank_summary_path,
    sep="\t",
    index=False,
    float_format="%.10g",
)

written_files.extend([
    blank_wells_path,
    blank_summary_path,
])

print(
    f"Processed {len(plate_pairs)} plate(s)."
)

print(
    "Conditions included:",
    condition_order,
)

print(
    "Primary OD600 column:",
    (
        "blank-corrected"
        if APPLY_BLANK_CORRECTION
        else "raw"
    ),
)

print("\nCreated files:")

for path in written_files:
    print(f"  {path}")

Processed 3 plate(s).
Conditions included: ['0x', '1x', '2x']
Primary OD600 column: blank-corrected

Created files:
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/CHR13_C_0x_input_long.tsv
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/CHR13_T_1x_input_long.tsv
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/CHR13_T_2x_input_long.tsv
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/all_plates_input_long.tsv
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/all_blank_wells.tsv
  /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/7_HIPHOP_validation/Single-dose_QTL_randomized/input/blank_summary.tsv


In [6]:
assert not combined_long.empty

required_output_columns = [
    "plate",
    "condition",
    "sample_id",
    "gene",
    "qtl",
    "replicate",
    "replicate_number",
    "well",
    "OD600",
    "OD600_raw",
    "blank_mean",
    "OD600_blank_corrected",
]

assert combined_long[
    required_output_columns
].notna().all().all()

assert not combined_long.duplicated(
    [
        "plate",
        "sample_id",
        "replicate",
    ]
).any()

numeric_values = combined_long[
    [
        "OD600",
        "OD600_raw",
        "blank_mean",
        "OD600_blank_corrected",
    ]
].to_numpy()

assert np.isfinite(
    numeric_values
).all()

recalculated_correction = (
    combined_long["OD600_raw"]
    - combined_long["blank_mean"]
)

assert np.allclose(
    combined_long[
        "OD600_blank_corrected"
    ],
    recalculated_correction,
    rtol=1e-10,
    atol=1e-10,
)

condition_summary = (
    combined_long
    .groupby(
        "condition",
        as_index=False,
    )
    .agg(
        plates=("plate", "nunique"),
        qtls=("qtl", "nunique"),
        strains=("sample_id", "nunique"),
        measurements=("OD600", "size"),
    )
)

replicate_counts = (
    combined_long
    .groupby(
        [
            "plate",
            "condition",
            "qtl",
            "sample_id",
            "gene",
        ],
        as_index=False,
    )
    .agg(
        replicates=("OD600", "size"),
        mean_OD600=("OD600", "mean"),
    )
)

print("Condition summary")
display(condition_summary)

print("Blank-well summary")
display(blank_summary)

print("Replicate counts")
display(replicate_counts)

print("Combined long-format preview")
display(combined_long.head(20))

print("All validation checks passed.")

Condition summary


,condition,plates,qtls,strains,measurements
0,0x,1,1,10,90
1,1x,1,1,10,90
2,2x,1,1,10,90


Blank-well summary


,plate,condition,blank_n,blank_mean,blank_median,blank_sd,blank_min,blank_max
0,CHR13_C_0x,0x,6,1.242000,0.4305,1.281414,0.399,3.036
1,CHR13_T_1x,1x,6,0.214167,0.1060,0.269864,0.100,0.765
2,CHR13_T_2x,2x,6,0.278500,0.1050,0.426463,0.102,1.149


Replicate counts


,plate,condition,qtl,sample_id,gene,replicates,mean_OD600
0,CHR13_C_0x,0x,all,1,TIF11,9,1.733667
1,CHR13_C_0x,0x,all,10,RSN1,9,1.642333
2,CHR13_C_0x,0x,all,2,TRM732,9,1.720000
3,CHR13_C_0x,0x,all,3,YMR262W,9,1.763333
4,CHR13_C_0x,0x,all,4,BUL1,9,1.605333
5,CHR13_C_0x,0x,all,5,PPA2,9,1.419667
6,CHR13_C_0x,0x,all,6,RRN9,9,1.623333
7,CHR13_C_0x,0x,all,7,SCS7,9,1.650333
8,CHR13_C_0x,0x,all,8,ZDS1,9,1.795333
9,CHR13_C_0x,0x,all,9,URA10,9,1.497333


Combined long-format preview


,plate,condition,sample_id,gene,qtl,replicate,replicate_number,well,OD600,OD600_raw,blank_mean,OD600_blank_corrected
0,CHR13_C_0x,0x,1,TIF11,all,replicate_01,1,A11,1.857,3.099,1.242,1.857
1,CHR13_C_0x,0x,1,TIF11,all,replicate_02,2,B5,1.248,2.490,1.242,1.248
2,CHR13_C_0x,0x,1,TIF11,all,replicate_03,3,C1,1.887,3.129,1.242,1.887
3,CHR13_C_0x,0x,1,TIF11,all,replicate_04,4,D8,1.749,2.991,1.242,1.749
4,CHR13_C_0x,0x,1,TIF11,all,replicate_05,5,E3,1.671,2.913,1.242,1.671
5,CHR13_C_0x,0x,1,TIF11,all,replicate_06,6,E12,1.401,2.643,1.242,1.401
6,CHR13_C_0x,0x,1,TIF11,all,replicate_07,7,F2,1.437,2.679,1.242,1.437
7,CHR13_C_0x,0x,1,TIF11,all,replicate_08,8,G6,1.662,2.904,1.242,1.662
8,CHR13_C_0x,0x,1,TIF11,all,replicate_09,9,H4,2.691,3.933,1.242,2.691
9,CHR13_C_0x,0x,10,RSN1,all,replicate_01,1,A10,1.992,3.234,1.242,1.992


All validation checks passed.
